# Calculate Regrowth Fractions for ΔrhlA P. aeruginosa

## Overview
This notebook calculates regrowth fractions across spatial gradients for all replicates of the ΔrhlA mutant and generates intermediate CSV files for downstream analysis and plotting.

## Analysis Workflow
1. **Calculate Mean Cell Density**: Compute average cells per spatial bin from initial fluorescence data
2. **Define Spatial Bins**: Create 10 × 10 grid of bins spanning 0-50 µm in x and y directions
3. **Calculate Regrowth Fractions**: Normalize colony counts by mean cell density (`b_mean`)
4. **Aggregate Across Levels**: 
   - Per chamber: Individual chamber-bin combinations
   - Per replicate: Average across chambers within each replicate
   - Final: Average across all three replicates

## Outputs
- `1_first_colonies_rhlA_merged.csv`: Combined colony data from all replicates
- `1_regrowth_fractions_rhlA_per_chamber.csv`: Regrowth fractions for each chamber-bin
- `1_regrowth_fractions_rhlA_per_replicate.csv`: Regrowth fractions averaged per replicate
- `1_regrowth_fractions_rhlA_merged.csv`: Final regrowth fractions with statistics across replicates

In [1]:
# Data analysis packages
import numpy as np
import pandas as pd

# Visualization packages
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
import matplotlib.cm as cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

# File handling
import os

# Verify working directory
print(f"Current working directory: {os.getcwd()}")
print(f"Expected to be in: .../Figure2/2A_mutants/2A_rhlA/figure_code/")

Current working directory: /Users/simonvanvliet/Library/CloudStorage/Dropbox/Work/Code/Spatial-Tolerance-Figure-Data-and-Code/Figure2/2A_mutants/2A_rhlA/figure_code
Expected to be in: .../Figure2/2A_mutants/2A_rhlA/figure_code/


# Step 1: Calculate Mean Cell Density per Bin

## Objective
Compute the average number of cells per spatial bin across all replicates and chambers. This value (`b_mean`) serves as the normalization factor for calculating regrowth fractions.

## Method
1. Load fluorescence-classified cell data from all three replicates
2. Extract cell centroids at frame 0 (initial time point)
3. Bin cells along y-axis into 8 bins spanning 0-40 µm
4. Count cells per bin per chamber
5. Average across chambers within each replicate
6. Average across all replicates

## Output
- `b_mean`: Mean cells per bin, used for normalization in regrowth fraction calculation

In [ ]:
# Configuration parameters
pixel_to_um = 0.065  # Pixel size in micrometers
frame_number = 0  # Initial time point
num_bins = 10  # Number of spatial bins
max_distance = 50  # Maximum distance in micrometers

# Define bin edges along y-axis
y_bins = np.linspace(0, max_distance, num_bins + 1)
bin_width = (y_bins[-1] - y_bins[0]) / num_bins

# Define replicate data sources (relative paths)
replicate_sources = [
    ('rep1', '../analysis_code/1_fluo_binary_rhlA_rep1.csv'),
    ('rep2', '../analysis_code/1_fluo_binary_rhlA_rep2.csv'),
    ('rep3', '../analysis_code/1_fluo_binary_rhlA_rep3.csv'),
]

# Load and process cell data from all replicates
cell_frames = []
for rep_label, csv_path in replicate_sources:
    df = pd.read_csv(csv_path)
    
    # Filter to frame 0 only
    df = df[df['frame_number'] == frame_number].copy()
    if df.empty:
        continue

    # Convert pixel coordinates to micrometers
    df['x_um'] = df['x'] * pixel_to_um
    df['y_um'] = df['y'] * pixel_to_um

    # Aggregate pixels to cell centroids (one row per cell)
    cell_df = (df
               .groupby(['pos', 'label'], as_index=False)
               .agg(x_um=('x_um', 'mean'),
                    y_um=('y_um', 'mean')))
    
    cell_df['replicate'] = rep_label
    cell_frames.append(cell_df)

# Combine all replicates
cells = pd.concat(cell_frames, ignore_index=True)

# Filter to analysis region (0-40 µm in both dimensions)
cells = cells[(cells['x_um'] >= 0) & (cells['x_um'] <= max_distance) & 
              (cells['y_um'] >= 0) & (cells['y_um'] <= max_distance)].copy()

print(f"Total cells analyzed: {len(cells)}")

# Assign cells to y-axis bins
cells['y_bin'] = pd.cut(cells['y_um'], bins=y_bins, include_lowest=True)
cells = cells.dropna(subset=['y_bin'])
cells['bin_left'] = cells['y_bin'].apply(lambda iv: float(iv.left))

# Count cells per chamber per bin
chamber_bin_counts = (cells
                      .groupby(['replicate', 'pos', 'bin_left'], as_index=False)
                      .size()
                      .rename(columns={'size': 'cell_count'}))

# Aggregate: Mean across chambers within each replicate
replicate_bin_stats = (chamber_bin_counts
                       .groupby(['replicate', 'bin_left'], as_index=False)
                       .agg(mean_cells_per_bin=('cell_count', 'mean')))

# Aggregate: Mean across replicates
bin_stats = (replicate_bin_stats
             .groupby('bin_left', as_index=False)
             .agg(mean_cells_per_bin=('mean_cells_per_bin', 'mean')))

bin_stats['bin_left'] = bin_stats['bin_left'].astype(float)

# Calculate overall mean and standard deviation
b_mean = bin_stats['mean_cells_per_bin'].mean()
b_std = bin_stats['mean_cells_per_bin'].std()

# Print summary
print(f"\nMean cells per bin (b_mean): {b_mean:.2f} ± {b_std:.2f}")
print(f"  Computed from {num_bins} bins across 3 replicates")
print(f"\nThis b_mean value will be used for regrowth fraction normalization.")

# Step 2: Define Spatial Bins for Colony Analysis

## Objective
Create a uniform 10 × 10 grid of spatial bins for analyzing colony positions.

## Configuration
- Number of bins: 10 in each dimension (x and y)
- Spatial range: 0-50 µm
- Bin width: 5 µm

## Output
- `x_bins`: Array of bin edges along x-axis
- `y_bins`: Array of bin edges along y-axis

In [ ]:
# Spatial binning parameters
pixel_to_um = 0.065  # Pixel size conversion factor
num_bins = 10  # Number of bins per axis
max_distance = 50  # Maximum distance in micrometers

# Create bin edges (evenly spaced from 0 to 50 µm)
x_bins = np.linspace(0, max_distance, num_bins + 1)
y_bins = np.linspace(0, max_distance, num_bins + 1)

# Calculate bin width
bin_width = (x_bins[1] - x_bins[0])

# Print configuration
print(f"Spatial binning configuration:")
print(f"  Number of bins per axis: {num_bins}")
print(f"  Spatial range: 0-{max_distance} µm")
print(f"  Bin width: {bin_width:.2f} µm")
print(f"\nBin edges (µm):")
print(f"  x_bins: {x_bins}")
print(f"  y_bins: {y_bins}")

Spatial binning configuration:
  Number of bins per axis: 10
  Spatial range: 0-50 µm
  Bin width: 5.00 µm

Bin edges (µm):
  x_bins: [ 0.  5. 10. 15. 20. 25. 30. 35. 40. 45. 50.]
  y_bins: [ 0.  5. 10. 15. 20. 25. 30. 35. 40. 45. 50.]


# Step 3: Calculate Regrowth Fractions

## Objective
Compute regrowth fractions for each spatial bin by normalizing colony counts with the mean cell density (`b_mean`).

## Workflow
1. **Load Colony Data**: Import first-colony detection data from all three replicates
2. **Filter Colonies**: 
   - Remove merged colonies (ID ≥ 10,000)
   - Restrict to first 20 hours of observation
3. **Spatial Binning**: Assign each colony to x and y bins
4. **Count Colonies**: Tabulate colonies per chamber per bin
5. **Calculate Regrowth Fractions**: Normalize by `b_mean` at three levels:
   - **Per chamber**: Individual regrowth fraction for each chamber-bin combination
   - **Per replicate**: Average across all chambers within a replicate (accounting for chambers with zero regrowth)
   - **Final**: Average across all three replicates with standard deviation

## Normalization Formula
$$\text{Regrowth Fraction} = \frac{\text{Colony Count}}{b_{\text{mean}}}$$

Where `b_mean` is the average number of cells per bin from the initial cell distribution.

## Outputs
- Per-chamber regrowth fractions
- Per-replicate regrowth fractions
- Merged regrowth fractions with statistics

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Define replicate data sources (relative paths to analysis outputs)
replicates = [
    {
        'colonies_path': '../analysis_code/2_first_colonies_rhlA_rep1.csv',
        'replicate': 'rep1'
    },
    {
        'colonies_path': '../analysis_code/2_first_colonies_rhlA_rep2.csv',
        'replicate': 'rep2'
    },
    {
        'colonies_path': '../analysis_code/2_first_colonies_rhlA_rep3.csv',
        'replicate': 'rep3'
    }
]

# Define output file paths
OUTPUT_COLONIES = '1_first_colonies_rhlA_merged.csv'
OUTPUT_REGROWTH_FINAL = '1_regrowth_fractions_rhlA_merged.csv'
OUTPUT_PER_REP_COMBINED = '1_regrowth_fractions_rhlA_per_replicate.csv'
OUTPUT_PER_CHAMBER = '1_regrowth_fractions_rhlA_per_chamber.csv'

print(f"Normalization factor (b_mean): {b_mean:.2f}\n")

# =============================================================================
# STEP 1: Compute Per-Chamber Regrowth Fractions
# =============================================================================

all_colonies = []
regrowth_per_chamber = []
total_chambers_per_replicate = {}

for rep in replicates:
    path = rep["colonies_path"]
    rep_name = rep["replicate"]

    # Check if file exists
    if not os.path.exists(path):
        print(f"⚠ Skipping {rep_name} (file not found: {path})")
        continue

    # Load colony data
    df = pd.read_csv(path)
    total_chambers = df["position"].nunique()
    total_chambers_per_replicate[rep_name] = total_chambers

    print(f"{rep_name}: {total_chambers} chambers analyzed")
    positions = sorted(df["position"].dropna().unique())
    print(f"  Positions: {', '.join(positions)}")
    print("-" * 80)

    # Filter to valid colonies
    colonies = df[df["colony_id"].notna()].copy()
    if colonies.empty:
        print(f"  {rep_name}: No regrowth detected\n")
        continue

    # Convert frames to hours and apply time/ID filters
    colonies["hours"] = colonies["frame"] * (5 / 60)  # 5 min/frame
    colonies = colonies[(colonies["hours"] <= 20) & (colonies["colony_id"] < 10000)].copy()
    
    if colonies.empty:
        print(f"  {rep_name}: No colonies after filtering\n")
        continue

    # Add metadata and convert to micrometers
    colonies["replicate"] = rep_name
    colonies["x_um"] = colonies["x"] * pixel_to_um
    colonies["y_um"] = colonies["y"] * pixel_to_um

    # Assign colonies to spatial bins
    colonies["x_bin"] = pd.cut(colonies["x_um"], bins=x_bins, include_lowest=True)
    colonies["y_bin"] = pd.cut(colonies["y_um"], bins=y_bins, include_lowest=True)

    # --- Count colonies per chamber per bin (Y-axis) ---
    chamber_bin_counts_y = (
        colonies.groupby(["position", "y_bin"])
        .size()
        .reset_index(name="colony_count")
        .assign(
            replicate=rep_name,
            axis="y",
            bin_left=lambda df: df["y_bin"].apply(lambda x: float(x.left) if pd.notna(x) else None)
        )
        .dropna(subset=["bin_left"])
    )
    # Clip negative bin edges to zero and calculate regrowth fraction
    chamber_bin_counts_y["bin_left"] = chamber_bin_counts_y["bin_left"].astype(float).clip(lower=0)
    chamber_bin_counts_y["regrowth_fraction"] = chamber_bin_counts_y["colony_count"] / b_mean

    # --- Count colonies per chamber per bin (X-axis) ---
    chamber_bin_counts_x = (
        colonies.groupby(["position", "x_bin"])
        .size()
        .reset_index(name="colony_count")
        .assign(
            replicate=rep_name,
            axis="x",
            bin_left=lambda df: df["x_bin"].apply(lambda x: float(x.left) if pd.notna(x) else None)
        )
        .dropna(subset=["bin_left"])
    )
    # Clip negative bin edges to zero and calculate regrowth fraction
    chamber_bin_counts_x["bin_left"] = chamber_bin_counts_x["bin_left"].astype(float).clip(lower=0)
    chamber_bin_counts_x["regrowth_fraction"] = chamber_bin_counts_x["colony_count"] / b_mean

    # Store per-chamber data
    regrowth_per_chamber.append(
        chamber_bin_counts_y[["replicate", "position", "axis", "bin_left", "colony_count", "regrowth_fraction"]]
    )
    regrowth_per_chamber.append(
        chamber_bin_counts_x[["replicate", "position", "axis", "bin_left", "colony_count", "regrowth_fraction"]]
    )

    # Store colony data
    all_colonies.append(colonies)

# Combine per-chamber data from all replicates
if regrowth_per_chamber:
    regrowth_per_chamber_df = pd.concat(regrowth_per_chamber, ignore_index=True)
else:
    regrowth_per_chamber_df = pd.DataFrame(
        columns=["replicate", "position", "axis", "bin_left", "colony_count", "regrowth_fraction"]
    )

# Save merged colonies data
merged_colonies_df = pd.concat(all_colonies, ignore_index=True) if all_colonies else pd.DataFrame()
merged_colonies_df.to_csv(OUTPUT_COLONIES, index=False)
print(f"\n✅ Saved merged colonies to: {OUTPUT_COLONIES}")

# Save per-chamber regrowth data
regrowth_per_chamber_df.to_csv(OUTPUT_PER_CHAMBER, index=False)
print(f"✅ Saved per-chamber regrowth fractions to: {OUTPUT_PER_CHAMBER}")

# =============================================================================
# STEP 2: Normalize Per Replicate (Include Chambers with Zero Regrowth)
# =============================================================================

# Sum regrowth fractions within each replicate-axis-bin combination
replicate_bin_stats = (
    regrowth_per_chamber_df
    .groupby(["replicate", "axis", "bin_left"], as_index=False)
    .agg(total_fraction_in_bin=("regrowth_fraction", "sum"))
)

# Normalize by total number of chambers (including those with zero regrowth)
replicate_bin_stats["total_chambers"] = replicate_bin_stats["replicate"].map(total_chambers_per_replicate)
replicate_bin_stats["regrowth_fraction"] = (
    replicate_bin_stats["total_fraction_in_bin"] / replicate_bin_stats["total_chambers"]
)

# Save per-replicate regrowth data
replicate_bin_stats[["replicate", "axis", "bin_left", "regrowth_fraction"]].to_csv(
    OUTPUT_PER_REP_COMBINED, index=False
)
print(f"✅ Saved per-replicate regrowth fractions to: {OUTPUT_PER_REP_COMBINED}")

# =============================================================================
# STEP 3: Average Across Replicates
# =============================================================================

n_replicates = len(replicates)

# Calculate mean and standard deviation across replicates
bin_stats = (
    replicate_bin_stats
    .groupby(["axis", "bin_left"], as_index=False)
    .agg(
        mean_regrowth_fraction=("regrowth_fraction", lambda x: x.sum() / n_replicates),
        std_regrowth_fraction=("regrowth_fraction", "std"),
        n_replicates=("replicate", "nunique")
    )
)

# Fill NaN standard deviations with zero (for bins with only one replicate)
bin_stats["std_regrowth_fraction"] = bin_stats["std_regrowth_fraction"].fillna(0)

# Save final merged regrowth data
bin_stats.to_csv(OUTPUT_REGROWTH_FINAL, index=False)
print(f"✅ Saved final regrowth fractions to: {OUTPUT_REGROWTH_FINAL}")

# =============================================================================
# PRINT SUMMARY
# =============================================================================

bin_width = x_bins[1] - x_bins[0]

print("\n" + "=" * 80)
print("REGROWTH FRACTION PER BIN (mean ± std across replicates)")
print("=" * 80)

# Print results for each axis
for axis in ["x", "y"]:
    print(f"\n{axis.upper()}-AXIS:")
    print("-" * 80)
    axis_stats = bin_stats[bin_stats["axis"] == axis].sort_values("bin_left")

    for _, row in axis_stats.iterrows():
        b0, b1 = row["bin_left"], row["bin_left"] + bin_width
        print(f"  Bin {b0:5.1f}-{b1:5.1f} µm: "
              f"{row['mean_regrowth_fraction']:.6f} ± {row['std_regrowth_fraction']:.6f} "
              f"(n={row['n_replicates']})")

    # Calculate overall statistics for this axis
    overall_mean = axis_stats["mean_regrowth_fraction"].mean()
    overall_std = axis_stats["mean_regrowth_fraction"].std()
    print(f"\n  Overall {axis}-axis mean: {overall_mean:.6f} ± {overall_std:.6f}")

# Print global summary
print("\n" + "=" * 80)
print("OVERALL SUMMARY")
print("=" * 80)
print(f"Total colonies detected: {len(merged_colonies_df)}")
print(f"Total chamber-bin combinations: {len(regrowth_per_chamber_df)}")
print(f"Mean regrowth fraction (all bins): "
      f"{bin_stats['mean_regrowth_fraction'].mean():.6f} ± "
      f"{bin_stats['std_regrowth_fraction'].std():.6f}")
print(f"Normalization factor (b_mean): {b_mean:.2f}")
print("=" * 80)

Normalization factor (b_mean): 156.94

rep1: 10 chambers analyzed
  Positions: pos0, pos1, pos2, pos3, pos4, pos5, pos6, pos7, pos8, pos9
--------------------------------------------------------------------------------
rep2: 9 chambers analyzed
  Positions: pos18, pos19, pos20, pos21, pos22, pos23, pos24, pos26, pos27
--------------------------------------------------------------------------------
rep3: 10 chambers analyzed
  Positions: pos18, pos19, pos20, pos21, pos22, pos23, pos24, pos25, pos26, pos27
--------------------------------------------------------------------------------

✅ Saved merged colonies to: 1_first_colonies_rhlA_merged.csv
✅ Saved per-chamber regrowth fractions to: 1_regrowth_fractions_rhlA_per_chamber.csv
✅ Saved per-replicate regrowth fractions to: 1_regrowth_fractions_rhlA_per_replicate.csv
✅ Saved final regrowth fractions to: 1_regrowth_fractions_rhlA_merged.csv

REGROWTH FRACTION PER BIN (mean ± std across replicates)

X-AXIS:
--------------------------------

In [ ]:
# ----------------------------------------------------------------------
# Data -----------------------------------------------------------------
try:
    colonies_df = pd.read_csv(
        '/Volumes/ScientificData/Users/Giulia(botgiu00)/Papers/bottacin2025/Figure_Data/Figure2/2A_mutants/2A_rhlA/4_first_colonies_rhlA_merged.csv'
    )
    # Use the FINAL merged file with mean across replicates
    regrowth_df = pd.read_csv(
        '/Volumes/ScientificData/Users/Giulia(botgiu00)/Papers/bottacin2025/Figure_Data/Figure2/2A_mutants/2A_rhlA/4_regrowth_fractions_rhlA_merged.csv'
    )
    print("Data loaded successfully!")
    print(f"Colonies: {len(colonies_df)} rows")
    print(f"Regrowth stats: {len(regrowth_df)} rows")
    print("\nRegrowth columns:", regrowth_df.columns.tolist())
except FileNotFoundError as e:
    print(f"Error loading data: {e}")
    raise

# ----------------------------------------------------------------------
# Constants -------------------------------------------------------------
num_bins      = 10
pnas_fontsize = 8
fig_width, fig_height = 3, 2.5   # inches
color_wt = '#cc99ff'
replicate_markers = {'rep1': 'o', 'rep2': 'o', 'rep3': 'o'}

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# Determine data range for plotting
x_max = max(colonies_df['x_um'].max(), colonies_df['y_um'].max())
x_max = np.ceil(x_max / 10) * 10  # Round up to nearest 10

# Calculate bin width from data
bin_width = x_max / num_bins

# ----------------------------------------------------------------------
# Figure & GridSpec -----------------------------------------------------
fig = plt.figure(figsize=(fig_width, fig_height))
gs = gridspec.GridSpec(
    2, 2,
    width_ratios=[1, 0.4],
    height_ratios=[2, 1]
)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1], sharey=ax1)
ax3 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax4 = fig.add_subplot(gs[1, 1])

# ----------------------------------------------------------------------
# Plotting --------------------------------------------------------------
# --- Plot 1: Scatter (rotated) ---
for rep, marker in replicate_markers.items():
    df_rep = colonies_df[colonies_df['replicate'] == rep]
    ax1.scatter(df_rep['y_um'], df_rep['x_um'],   # swapped to rotate
                marker=marker, label=rep,
                color=color_wt, s=15, alpha=0.2,
                edgecolors='k', linewidths=0.1)

ax1.set_aspect('equal', adjustable='box', anchor='W')
ax1.set_xlim(0, x_max); ax1.set_ylim(0, x_max)
ax1.set_xlabel('Distance to PA (µm)', fontsize=pnas_fontsize)
ax1.set_ylabel('Chamber Width (µm)', fontsize=pnas_fontsize)
ax1.grid(False)
ax1.set_title(r'$ \it{P.\ aeruginosa}\ \mathrm{wt}$', fontsize=pnas_fontsize)

# Flip horizontally (mirror along x-axis)
ax1.invert_yaxis()

# Helper to linearly extend first/last bin-center values to the edges
def extend_to_edges(centers, values, left_edge=0.0, right_edge=None):
    if right_edge is None:
        right_edge = x_max
    centers = np.asarray(centers)
    values  = np.asarray(values)
    if len(centers) < 2:
        x = np.array([left_edge, centers[0], right_edge])
        y = np.array([values[0], values[0], values[0]])
        return x, y
    # Left extrapolation
    m_left   = (values[1] - values[0]) / (centers[1] - centers[0])
    left_val = values[0] - m_left * (centers[0] - left_edge)
    # Right extrapolation
    m_right   = (values[-1] - values[-2]) / (centers[-1] - centers[-2])
    right_val = values[-1] + m_right * (right_edge - centers[-1])
    x = np.concatenate(([left_edge], centers, [right_edge]))
    y = np.concatenate(([left_val],  values,  [right_val]))
    return x, y

# --- Plot 2: X-axis regrowth (horizontal) ---
# Columns: axis, bin_left, mean_regrowth_fraction, std_regrowth_fraction, n_replicates
reg_x = regrowth_df[regrowth_df['axis'] == 'x'].copy()
reg_x = reg_x.sort_values('bin_left')

centers_x = reg_x['bin_left'].values + bin_width / 2.0
means_x   = reg_x['mean_regrowth_fraction'].values
stds_x    = reg_x['std_regrowth_fraction'].values

# Extend to edges
y_line_x, mean_line_x = extend_to_edges(centers_x, means_x, left_edge=0.0, right_edge=x_max)
_, std_line_x = extend_to_edges(centers_x, stds_x, left_edge=0.0, right_edge=x_max)
std_line_x[0]  = stds_x[0]
std_line_x[-1] = stds_x[-1]

ax2.plot(mean_line_x, y_line_x, color=color_wt, linewidth=1)
ax2.fill_betweenx(y_line_x, mean_line_x - std_line_x, mean_line_x + std_line_x,
                  color=color_wt, alpha=0.3)

# Determine x-axis limits for regrowth probability
max_regrowth = max(means_x.max() + stds_x.max(), 0.1)
max_regrowth = np.ceil(max_regrowth * 10) / 10  # Round up to nearest 0.1

ax2.set_ylabel('Regrowth probability', fontsize=pnas_fontsize, rotation=-90, labelpad=-45)
ax2.set_xlim(0, 0.075); ax2.set_ylim(0, x_max)
ax2.set_yticks([])
ax2.set_xticks([0.025, 0.05, 0.075])
ax2.xaxis.set_label_position('top')
ax2.xaxis.tick_top()
ax2.tick_params(axis='x', labelsize=pnas_fontsize)
ax2.set_xticklabels(ax2.get_xticks(), rotation=-45, ha='right')
ax2.grid(False)

# Spine
ax2.spines['top'].set_visible(True)
ax2.spines['bottom'].set_visible(False)

# --- Plot 3: Y-axis regrowth (vertical) ---
reg_y = regrowth_df[regrowth_df['axis'] == 'y'].copy()
reg_y = reg_y.sort_values('bin_left')

centers_y = reg_y['bin_left'].values + bin_width / 2.0
means_y   = reg_y['mean_regrowth_fraction'].values
stds_y    = reg_y['std_regrowth_fraction'].values

x_line_y, mean_line_y = extend_to_edges(centers_y, means_y, left_edge=0.0, right_edge=x_max)
_, std_line_y = extend_to_edges(centers_y, stds_y, left_edge=0.0, right_edge=x_max)
std_line_y[0]  = stds_y[0]
std_line_y[-1] = stds_y[-1]

ax3.plot(x_line_y, mean_line_y, color=color_wt, linewidth=1)
ax3.fill_between(x_line_y, mean_line_y - std_line_y, mean_line_y + std_line_y,
                 color=color_wt, alpha=0.3)

ax3.set_xlabel('Regrowth probability', fontsize=pnas_fontsize, labelpad=-50)
ax3.set_xlim(0, x_max); ax3.set_ylim(0, 0.075)
ax3.set_xticks([])
ax3.set_yticks([0.025, 0.05, 0.075])
ax3.tick_params(axis='y', labelsize=pnas_fontsize, direction='out', length=4, width=1)
ax3.grid(False)

# --- Plot 4: Legend ---
ax4.axis('off')

# ----------------------------------------------------------------------
# Manual Alignment ------------------------------------------------------
fig.canvas.draw()

w_gap_in = 0.05
h_gap_in = 0.05
w_gap_fig = w_gap_in / fig_width
h_gap_fig = h_gap_in / fig_height

bbox1_data = ax1.get_position()
bbox2_full = ax2.get_position()
bbox3_full = ax3.get_position()
bbox4_full = ax4.get_position()

# Align ax3 ABOVE ax1
ax3.set_position([
    bbox1_data.x0,
    bbox1_data.y1 + h_gap_fig,
    bbox1_data.width,
    bbox3_full.height
])

# Align ax2
ax2.set_position([
    bbox1_data.x0 + bbox1_data.width + w_gap_fig,
    bbox1_data.y0,
    bbox2_full.width,
    bbox1_data.height
])

# Align ax4
bbox2_new = ax2.get_position()
ax4.set_position([
    bbox2_new.x0,
    bbox1_data.y0 - bbox4_full.height - h_gap_fig,
    bbox2_new.width,
    bbox4_full.height
])

# ----------------------------------------------------------------------
# Cosmetics -------------------------------------------------------------
for ax in (ax1, ax2, ax3):
    ax.tick_params(axis='both', labelsize=pnas_fontsize)

ax3.tick_params(axis='y', which='both', left=True, labelleft=True)

# Save figure
#output_path = '/Volumes/ScientificData/Users/Giulia(botgiu00)/Papers/bottacin2025/Figure_Data/Figure2/2A_mutants/2A_rhlA/2AS_rhlA.pdf'
#plt.savefig(output_path, format='pdf', bbox_inches='tight', dpi=300)
#print(f"\nFigure saved: {output_path}")

plt.show()